# Kernel cuantico para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map elegido en `FEATURE_MAP`
(ZZ, Pauli Z+YY o Ry-CX-Rx) definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> enviar y guardar
K_train (para `SVC.fit`) -> enviar y guardar K_test (para `SVC.predict`).
Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los parametros
y las llamadas.

La construccion esta apagada por defecto (`RUN_MATRIX = False`): revisa
circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado y lo separa en train/test segun `_PartInd_`.

In [14]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

# Los valores del kernel suelen ser pequenos; se muestran con 6 decimales
# fijos (sin notacion cientifica) para poder distinguirlos.
pd.set_option("display.float_format", "{:.6f}".format)

from funciones_nexus import (
    cargar_datos_kernel,
    obtener_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    iniciar_matriz_kernel_test,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    FEATURE_MAPS,
    MATRIX_BACKEND_OPTIONS,
)

# Feature map del kernel: elige una de las 3 opciones de FEATURE_MAPS.
#   "zz"       -> ZZFeatureMap (fases Z + interacciones ZZ)
#   "zyy"      -> Pauli Z+YY explicito (entrelazamiento lineal)
#   "ry_cx_rx" -> Ry -> cadena CX -> Rx
# El mismo FEATURE_MAP gobierna K_train y K_test (deben coincidir).
FEATURE_MAP = "zz"

PROJECT_NAME = "prueba_migracion"                       # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/df_escalado.csv"     # Dataset escalado con _PartInd_

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel(KERNEL_DATA_PATH)
print("Feature map:", FEATURE_MAP, "| opciones:", list(FEATURE_MAPS))
print("Features:", kernel_feature_columns)
print(f"Train: {kernel_train_df.shape} | Test: {kernel_test_df.shape}")
display(kernel_train_df.head())

ImportError: cannot import name 'obtener_feature_map' from 'funciones_nexus' (d:\quantathon\main\funciones_nexus.py)

## 2. Inspeccion del feature map U(x)

No consume shots.

In [ ]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_fn = obtener_feature_map(FEATURE_MAP)
feature_map_preview = feature_map_fn(preview_x)
print(f"Feature map '{FEATURE_MAP}' de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [ ]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J, feature_map=FEATURE_MAP
)
render_circuit_jupyter(kernel_preview_circuit)

## 4. Enviar la matriz K_train

`K_train = K(X_train, X_train)` (cuadrada, para `SVC.fit`). Se ejecuta el
triangulo superior y se refleja por simetria; para `m` filas de train se
requieren `m(m-1)/2` circuitos (mas la diagonal si `MATRIX_EXECUTE_DIAGONAL`).

Backends: Selene local o Nexus (Selene, H1/H2 via compile job, Helios). En
local el resultado llega aqui mismo; en Nexus se envia el job y se sigue en
el paso 5. **Tras enviar a Nexus no reejecutes esta celda.**

In [ ]:
MATRIX_ROWS = [0, 1, 2, 3]                   # Filas de train (definen K_train y las columnas de K_test)
MATRIX_BACKEND = "H2-EMULATOR"               # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = True                            # Interruptor de seguridad
MATRIX_SHOTS = 10
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # False fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

matrix_state_train, matrix_result_train = iniciar_matriz_kernel(
    kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
    n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_train is not None:
    display(pd.DataFrame(matrix_result_train["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_train["run_summary"])

## 5. Consultar y guardar K_train

En Nexus, reejecuta **solo esta celda** hasta que el job llegue a COMPLETED
(H1/H2 encadena compile -> execute automaticamente). En local no hay nada que
consultar (`matrix_state_train` es None) y se guarda directo la matriz del
paso 4. Persiste la matriz de Gram cuadrada como `kernel_qsvm_<run_id>.csv`
(lista para `SVC(kernel="precomputed")`) + metadatos.

In [ ]:
matrix_state_train, matrix_train_remoto = consultar_matriz_nexus(matrix_state_train, guardar=SAVE_MATRIX_RUN)

# Toma la matriz remota si ya llego; si no, la local del paso 4.
if matrix_train_remoto is not None:
    K_train_final = matrix_train_remoto
    fuente_train = f"nexus_{MATRIX_BACKEND}_train"
    id_train = matrix_state_train["job_ref"].id
elif matrix_result_train is not None:
    K_train_final = matrix_result_train
    fuente_train = "local_statevector_train"
    id_train = None
else:
    K_train_final = None
    print("K_train aun no disponible (job remoto en curso o sin construir).")

if K_train_final is not None:
    ruta_train, ruta_train_meta = guardar_kernel_qsvm(K_train_final, source=fuente_train, job_id=id_train)
    print("K_train guardada en:", ruta_train)
    print("Metadatos en:", ruta_train_meta)
    display(pd.read_csv(ruta_train, sep=";", index_col=0))

## 6. Enviar la matriz K_test

`K_test = K(X_test, X_train)` (rectangular n_test x m, para `SVC.predict`).
Reutiliza los mismos parametros del paso 4 (backend, shots, feature map) y las
mismas `MATRIX_ROWS` como columnas. Internamente apila `[test, train]`,
construye la conjunta y recorta el bloque test x train.

Es un **job independiente** del de K_train: en Nexus puedes lanzar este envio
sin esperar a que termine el de train, y consultar cada uno por separado.

In [ ]:
TEST_ROWS = [0, 1, 2, 3]                     # Filas de test (filas de K_test); columnas = MATRIX_ROWS

matrix_state_test, matrix_result_test = iniciar_matriz_kernel_test(
    kernel_train_df, MATRIX_ROWS, kernel_test_df, MATRIX_BACKEND, RUN_MATRIX,
    test_rows=TEST_ROWS, n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_test is not None:
    display(pd.DataFrame(matrix_result_test["kernel_matrix"], index=TEST_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_test["run_summary"])

## 7. Consultar y guardar K_test

Igual que el paso 5 pero para el job de test. Persiste la matriz rectangular
como `kernel_qsvm_test_<run_id>.csv` (filas = test, columnas = train). Con
K_train (paso 5) y K_test (aqui) ya tienes ambos artefactos para el QSVM:
`SVC(kernel="precomputed").fit(K_train, y_train).predict(K_test)`.

In [ ]:
matrix_state_test, matrix_test_remoto = consultar_matriz_nexus(matrix_state_test, guardar=SAVE_MATRIX_RUN)

if matrix_test_remoto is not None:
    K_test_final = matrix_test_remoto
    fuente_test = f"nexus_{MATRIX_BACKEND}_test"
    id_test = matrix_state_test["job_ref"].id
elif matrix_result_test is not None:
    K_test_final = matrix_result_test
    fuente_test = "local_statevector_test"
    id_test = None
else:
    K_test_final = None
    print("K_test aun no disponible (job remoto en curso o sin construir).")

if K_test_final is not None:
    ruta_test, ruta_test_meta = guardar_kernel_qsvm(K_test_final, source=fuente_test, job_id=id_test)
    print("K_test guardada en:", ruta_test)
    print("Metadatos en:", ruta_test_meta)
    display(pd.read_csv(ruta_test, sep=";", index_col=0))